#### Task:

Load all files in DataLoader class

example file name:

>    450_2025_5_14_0:45_LinearSteps_data_13599.csv

path to file:

>    /home/gishb/datasets/change_points/syth

# Import libraries

In [1]:
import pandas as pd
import numpy as np
import os
import random
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm
from IPython.display import clear_output

# Define global args

In [2]:
path_to_files_dir = "/home/gishb/datasets/change_points/training"
BATCH_SIZE = 128
NUM_WORKERS = 5

# Define ClassLoader

In [3]:
class TimeSeries(Dataset):
    def __init__(self, path_to_files_dir, sequence_length_nn: int = 72):
        self.path_to_files_dir = path_to_files_dir
        self.sequence_length_nn = sequence_length_nn
        self.list_of_files = self.__load_list_of_files()

    def __load_list_of_files(self) -> list:
        return os.listdir(self.path_to_files_dir)

    def __load_data(self, idx) -> pd.DataFrame:
        return pd.read_csv(self.path_to_files_dir+"/"+self.list_of_files[idx])

    def __len__(self):
        return len(os.listdir(self.path_to_files_dir))
        
    def __getitem__(self, idx):
        dataframe = self.__load_data(idx)
        
        x = dataframe["cps"].to_numpy()
        y = dataframe["values"].to_numpy()
        
        x = self.normalize_by_minmax(x)
        
        padded_x = self.padding_sequence(x, d_model=self.sequence_length_nn)
        padded_y = self.padding_sequence(y, d_model=self.sequence_length_nn)
        
        x_prepaired = self.prepaire_sequence_to_encoding(padded_x)
        y_prepaired = self.prepaire_sequence_to_encoding(padded_y)
        return torch.Tensor(x_prepaired), torch.Tensor(y_prepaired)

    def padding_sequence(self, data: np.array, d_model: int) -> np.array:
        not_dev_count = data.shape[0] % d_model
        if not_dev_count != 0:
            val_new_rows = int(np.ceil(data.shape[0] / d_model) * d_model - data.shape[0])
            data = np.pad(data, 
                     pad_width=((0), (val_new_rows)),  # Pad val_new_rows to sequence
                     mode='constant', 
                     constant_values=data[-1])
        return data
    
    def prepaire_sequence_to_encoding(self, data) -> torch.Tensor:
        """ Take padded numpy array to preapre it to torch.Tensor for NN
        """
        number_of_words = data.shape[0] // self.sequence_length_nn
        return torch.Tensor(data).view(number_of_words, self.sequence_length_nn)
    
    def normalize_by_minmax(self, data: np.array) -> np.array:
        return (data-data.min())/(data.max()-data.min())

# Create Dataset

In [4]:
dataset_creator = TimeSeries(path_to_files_dir=path_to_files_dir)

In [5]:
train_loader = DataLoader(dataset_creator, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

In [6]:
test = next(iter(train_loader))

In [7]:
test[0].shape

torch.Size([128, 7, 72])